# RCIS Scheduler
- Uses a FCFS (First Come, First Serve) Algorithm to determine interviewee bookings for companies

## Algo Approach
- For each day, get companies and interviewees available
- For each company in that day, get all timeslots where company is available.
- Filter interviewee list (form responses) for each timeslot + company preferred degree program, order based on response time
- Get interviewee at the top, remove interviewee from list


## Dependency Installation

## Imports

In [36]:
from datetime import datetime
from typing import NewType, Protocol
from enum import Enum
import pandas as pd

## Classes and Protocols for Scheduler

In [37]:
class DegreeProgram(str, Enum):
    ChemicalEngg = 'BS Chemical Engineering'
    CivilEngg = 'BS Civil Engineering'
    ComputerEngg = 'BS Computer Engineering'
    ComputerScience = 'BS Computer Science'
    ElectricalEngg = 'BS Electrical Engineering'
    ElectronicsEngg = 'BS Electronics Engineering'
    GeodeticEngg = 'BS Geodetic Engineering'
    IndustrialEngg = 'BS Industrial Engineering'
    MaterialsEngg = 'BS Materials Engineering'
    MechEngg = 'BS Mechanical Engineering'
    MetalEngg = 'BS Metallurgical Engineering'
    MiningEngg = 'BS Mining Engineering'

class Times(str, Enum): # not ideal, but I don't really want to deal with datetime right now
# Interview Simulations
    Time0900_0945 = "9:00-9:45 AM"
    Time1000_1045 = "10:00-10:45 AM"
    Time1100_1145 = "11:00-11:45 AM"
    Time1315_1400 = "1:15-2:00 PM"
    Time1415_1500 = "2:15-3:00 PM"
    Time1515_1600 = "3:15-4:00 PM"
    Time1615_1700 = "4:15-5:00 PM"

# Resume Consultations
    Time0900_0930 = "9:00-9:30 AM"
    Time0930_1000 = "9:30-10:00 AM"
    Time1015_1045 = "10:15-10:45 AM"
    Time1045_1115 = "10:45-11:15 AM"
    Time1130_1200 = "11:30 AM-12:00 PM"
    Time1330_1400 = "1:30-2:00 PM"
    Time1400_1430 = "2:00-2:30 PM"
    Time1445_1515 = "2:45-3:15 PM"
    Time1515_1545 = "3:15-3:45 PM"
    Time1600_1630 = "4:00-4:30 PM"
    Time1630_1700 = "4:30-5:00 PM"
class Participant(Protocol):
    @property
    def name(self) -> str:
        # Return str that is the name of the participant (could be interviewee/interviewer)
        ...
    @property
    def time_slots(self) -> list[Times]:
        # Returns time slots of participant
        ...
class Interviewee:
    def __init__(self, name: str, email: str, capes_id: str):
        self._name = name
        self._email = email
        self._capes_id = capes_id
        self._booked_times: dict[str, dict[str,list[Times]]] = {"November 13" : {"RC" : [], "IS" : []}, "November 14" : {"RC" : [], "IS" : []}}
    def __eq__(self, b):
        return type(self) == type(b) and self.name == b.name 
    def __hash__(self):
        return hash(self.name)
    def __str__(self):
        return f"Interviewee: {self._name}"
    @property
    def name(self) -> str:
        return self._name
    @property
    def email(self) -> str:
        return self._email
    @property
    def capes_id(self) -> str:
        return self._capes_id
    def add_time(self, time : Times, type : str, day : str):
        self._booked_times[day][type].append(time)
    def check_coinciding_times(self, time : Times, type : str, day : str):
        return 0 if time in self._booked_times[day]['RC' if type == 'IS' else 'IS'] else 1
    

class Interviewer:
    def __init__(self, name: str, time_slots: list[Times], degree_program_preference: list[DegreeProgram] | DegreeProgram, appointment_type : str, interviewer_capacity_per_timeslot : int = 2):
        self._name = name
        self._time_slots = time_slots
        self._degree_program_preference = degree_program_preference
        self._interviewee_list : dict[Times, list[Interviewee]] = dict()
        self._appointment_type = appointment_type
        self._interviewee_capacity_per_timeslot = interviewer_capacity_per_timeslot
    def __lt__(a : Interviewer, b : Interviewer):
        return a.priority < b.priority
    def unallocated_type(self, type):
        self._unallocated_type = type
    @property
    def name(self) -> str:
        return self._name
    @property
    def time_slots(self) -> list[Times]:
        return self._time_slots
    @property
    def degree_program_preference(self) -> list[DegreeProgram] | DegreeProgram:
        return self._degree_program_preference
    @property
    def interviewee_list(self) -> dict[Times, list[Interviewee]]:
        return self._interviewee_list
    @property
    def priority(self):
        return len(self._time_slots)
    @property
    def appointment_type(self):
        return self._appointment_type
    @property
    def interviewee_capacity_per_timeslot(self):
        return self._interviewee_capacity_per_timeslot
    def add_to_interviewee_list(self, interviewee: Interviewee, time : Times):
        if time not in self._time_slots:
            return 0
        if time not in self._interviewee_list.keys():
            self._interviewee_list[time] = []
        print(self._interviewee_list)
        if len(self._interviewee_list[time]) >= self._interviewee_capacity_per_timeslot:
            return 0
        self._interviewee_list[time].append(interviewee)
        return 1    
    

## Get Interviewe/Response Data

In [38]:
responses_url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vR5hzOahI90d182BRnnk_lF68X3GCWfbr6rpAIeds1ezrABvujUhMS9JN_maoTAtiJ11qx2eCf6SyIL/pub?gid=1183642779&single=true&output=csv"
capes_card_url_2526 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQa39d0BnFMf0GB_NdAHbON6clsrSVolPytpl5JJQRFG02kI_l5cZZbRhQfwvFj0L-ukZRNxi5AEW7A/pub?gid=0&single=true&output=csv"
capes_card_url_2425 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQa39d0BnFMf0GB_NdAHbON6clsrSVolPytpl5JJQRFG02kI_l5cZZbRhQfwvFj0L-ukZRNxi5AEW7A/pub?gid=978666763&single=true&output=csv"

interviewee_df = pd.read_csv(responses_url)
capes_card_df = pd.read_csv(capes_card_url_2526)
capes_card_2425_df = pd.read_csv(capes_card_url_2425)

capes_card_df = pd.concat([capes_card_df, capes_card_2425_df[['CAPES CARD', 'LAST NAME', 'FIRST NAME', 'MI', 'COURSE','YR STANDING']]]).drop_duplicates()

In [39]:
capes_card_df.head()

,CAPES CARD,LAST NAME,FIRST NAME,MI,COURSE,YR STANDING
0,jachua,Chua,John Victor,A.,BS Industrial Engineering,4th
1,tlarcalas,Arcalas,Terence,L.,BS HE,3rd
2,tsngo,Ngo,Trei Sebastian,S.,BS Industrial Engineering,3rd
3,amtolentino,Tolentino,Andrea Nichole,M.,BS Computer Engineering,3rd
4,aamatsuzaki,Matsuzaki,Aryssa,A.,BS Computer Engineering,3rd


## Data Cleaning

### Column Selection for Cleaning

In [40]:
# RC/IS date column name
date_column_filter = 'What is your most preferred date?'



In [41]:
# Get columns to clean
int_cols = interviewee_df.select_dtypes(include='float').columns
rcis_date_cols = interviewee_df.columns[interviewee_df.columns.str.contains(date_column_filter)]

# Remove NaN values
interviewee_df[int_cols] = interviewee_df[int_cols].fillna(12).astype('int') 
interviewee_df[rcis_date_cols] = interviewee_df[rcis_date_cols].fillna('None')
interviewee_df['Degree Program'] = interviewee_df['CAPES ID'].map(capes_card_df.set_index('CAPES CARD')['COURSE'])
interviewee_df['Name'] = interviewee_df['CAPES ID'].map(capes_card_df.set_index('CAPES CARD')['FIRST NAME']) + ' ' + interviewee_df['CAPES ID'].map(capes_card_df.set_index('CAPES CARD')['LAST NAME']) 

# sorry tinamad
old_string = 'What are your preferred time slots for RC, November 13 (Thursday)? '
new_string = 'RC Slot November 13 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)

old_string = 'What are your preferred time slots for IS, November 13 (Thursday)? '
new_string = 'IS Slot November 13 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)

old_string = 'What are your preferred time slots for RC, November 14 (Friday)? '
new_string = 'RC Slot November 14 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)

old_string = 'What are your preferred time slots for IS, November 14 (Friday)? '
new_string = 'IS Slot November 14 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)


## Scheduler

- Sort interviewees by increasing rank and timestamp for a certain day and timeslot
- Get a company
- Filter out interviewee list based on company requirements (course, etc.)
- FCFS picking
- Remove chosen interviewees from list

In [42]:
rc_date_filter = ['RC, November 13', 'RC, November 14']
is_date_filter = ['IS, November 13', 'IS, November 14']

columns = interviewee_df.columns[interviewee_df.columns.str.contains(is_date_filter[0])]

interviewee_df = interviewee_df.sort_values(by=['Timestamp'], ascending=True)

interviewee_df

,Timestamp,Data Privacy Agreement,CAPES ID,Are you representing any UPD College of Engineering Organization in joining this event?,Updated Resume or CV,What sub-event would you like to register for?,What is your most preferred date? [Resume Consultation (RC)],What is your most preferred date? [Interview Simulation (IS)],RC Slot November 13 - [9:00-9:30 AM],RC Slot November 13 - [9:30-10:00 AM],...,IS Slot November 13 - [4:15-5:00 PM],IS Slot November 14 - [9:00-9:45 AM],IS Slot November 14 - [10:00-10:45 AM],IS Slot November 14 - [11:00-11:45 AM],IS Slot November 14 - [1:15-2:00 PM],IS Slot November 14 - [2:15-3:00 PM],IS Slot November 14 - [3:15-4:00 PM],IS Slot November 14 - [4:15-5:00 PM],Degree Program,Name
0,11/2/2025 20:03:04,I agree,xmjaudalso,No,https://drive.google.com/open?id=1MqWge49TDvC0...,Resume Consultations (RC) ONLY,November 13 (Thu),None,12,12,...,12,12,12,12,12,12,12,12,BS Civil Engineering,Xander Aia Danina Jaudalso
1,11/4/2025 16:38:15,I agree,cecarpio,No,https://drive.google.com/open?id=1J7RqQFHnOZMs...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,12,7,6,5,4,1,2,3,BS Electronics Engineering,Cyress Lein Carpio
2,11/6/2025 0:57:43,I agree,mdpunzalan,No,https://drive.google.com/open?id=1gkdZswnlbqPg...,Both (RC & IS),November 13 (Thu),November 13 (Thu),12,12,...,2,1,12,12,12,12,12,12,BS Computer Engineering,Marianne Veronica Punzalan
3,11/6/2025 12:33:38,I agree,mmurillo,No,https://drive.google.com/open?id=18hwpPcI2noW1...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,12,7,6,1,2,3,4,5,BS Materials Engineering,Maria Romela Murillo
4,11/6/2025 14:42:04,I agree,cmborrega,No,https://drive.google.com/open?id=1eND6TnRl__uo...,Both (RC & IS),November 13 (Thu),November 13 (Thu),12,12,...,12,1,2,3,4,12,12,12,BS Computer Engineering,Clyde Lawrence Borrega
5,11/6/2025 16:08:45,I agree,lchernandez,No,https://drive.google.com/open?id=1r2ipJRzMnMrg...,Resume Consultations (RC) ONLY,November 13 (Thu),None,9,8,...,7,1,2,3,4,5,6,7,BS Industrial Engineering,Leevan Hernandez
6,11/6/2025 17:08:06,I agree,jglao,No,https://drive.google.com/open?id=1F-eUumjv0JR0...,Both (RC & IS),November 14 (Fri),November 14 (Fri),11,10,...,4,7,6,5,1,3,4,2,BS Industrial Engineering,John Erickson Lao
7,11/6/2025 19:36:57,I agree,jlambrocio,No,https://drive.google.com/open?id=1U96ChPRn0bBy...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,12,12,12,5,1,3,4,2,BS Industrial Engineering,John Rustly Mark Ambrocio


### Create Interviewer Dictionary for to Denote Availability

In [43]:
interviewer_availability_dict : dict[str, dict[str, list[Interviewer]]]= {
    'November 13' : {
        'RC' : [
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='RC'
            ),
            Interviewer(
                name='GHD',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                    Times.Time1330_1400,
                    Times.Time1400_1430,
                    Times.Time1445_1515,
                    Times.Time1515_1545,
                    Times.Time1600_1630,
                    Times.Time1630_1700
                ],
                degree_program_preference=[
                    DegreeProgram.CivilEngg,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.GeodeticEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MiningEngg
                ],
                appointment_type='RC'
            )
        ],
        'IS' : [
            Interviewer(
                name='GHD',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                    Times.Time1315_1400, 
                    Times.Time1415_1500,
                    Times.Time1515_1600,
                    Times.Time1615_1700,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='IS',
                interviewer_capacity_per_timeslot=1
            ),
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='IS'
            )
        ]
    },
    'November 14' : {
        'RC' : [
            Interviewer(
                name='Seven Seven Global Services Inc.',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                    Times.Time1330_1400,
                    Times.Time1400_1430,
                    Times.Time1445_1515,
                    Times.Time1515_1545,
                    Times.Time1600_1630,
                    Times.Time1630_1700
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='RC'
            ),
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='RC'
            ),
            Interviewer(
                name='Concepcion Industrial Corporation',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                    Times.Time1330_1400,
                    Times.Time1400_1430,
                    Times.Time1445_1515,
                    Times.Time1515_1545,
                    Times.Time1600_1630,
                    Times.Time1630_1700
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='RC'
            )    
        ],
        'IS' : [
            Interviewer(
                name='Seven Seven Global Services Inc.',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                    Times.Time1315_1400, 
                    Times.Time1415_1500,
                    Times.Time1515_1600,
                    Times.Time1615_1700,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='IS',
                interviewer_capacity_per_timeslot=1
            ),
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='IS'
            ),
            
        ]
    }
}

## November 13 RC

### Book Interviewees to Timeslots for RC

In [44]:
# filter_first_day_rc = interviewee_df[interviewee_df['What is your most preferred date?  [Resume Consultation (RC)]'].str.contains('November 13')]
filter_first_day_rc = interviewee_df.sort_values(by='What is your most preferred date?  [Resume Consultation (RC)]', key= lambda col : col == 'November 13')


### Sanity Check Logging RC and IS

In [45]:
sanity_check_df = filter_first_day_rc[filter_first_day_rc['Name'].str.contains('Leevan Hernandez')]
scores_series = sanity_check_df[sanity_check_df.columns[sanity_check_df.columns.str.contains(f'RC Slot November 13')]]
melted_df = scores_series.melt(
    ignore_index=False,
    var_name='Time Slot Header',
    value_name='Ranking'
).reset_index(drop=True)
melted_df

,Time Slot Header,Ranking
0,RC Slot November 13 - [9:00-9:30 AM],9
1,RC Slot November 13 - [9:30-10:00 AM],8
2,RC Slot November 13 - [10:15-10:45 AM],7
3,RC Slot November 13 - [10:45-11:15 AM],6
4,RC Slot November 13 - [11:30 AM-12:00 PM],5
5,RC Slot November 13 - [1:30-2:00 PM],4
6,RC Slot November 13 - [2:00-2:30 PM],3
7,RC Slot November 13 - [2:45-3:15 PM],11
8,RC Slot November 13 - [3:15-3:45 PM],10
9,RC Slot November 13 - [4:00-4:30 PM],2


In [46]:
sanity_check_df = filter_first_day_rc[filter_first_day_rc['Name'].str.contains('Leevan Hernandez')]
scores_series = sanity_check_df[sanity_check_df.columns[sanity_check_df.columns.str.contains(f'IS Slot November 13')]]
melted_df = scores_series.melt(
    ignore_index=False,
    var_name='Time Slot Header',
    value_name='Ranking'
).reset_index(drop=True)
melted_df

,Time Slot Header,Ranking
0,IS Slot November 13 - [9:00-9:45 AM],3
1,IS Slot November 13 - [10:00-10:45 AM],4
2,IS Slot November 13 - [11:00-11:45 AM],5
3,IS Slot November 13 - [1:15-2:00 PM],6
4,IS Slot November 13 - [2:15-3:00 PM],1
5,IS Slot November 13 - [3:15-4:00 PM],2
6,IS Slot November 13 - [4:15-5:00 PM],7


### Algo Function

In [47]:
def get_booked_interviewee(name : str, interviewees : list[Interviewee]):
    for x in interviewees:
        if x.name == name:
            return x

def scheduling_algorithm(day: str, type: str, filtered_df: pd.DataFrame, booked_interviewees : list[Interviewee] = []):
    for interviewer in interviewer_availability_dict[day][type]:
        filter_by_desired_course = filtered_df[filtered_df['Degree Program'].isin(interviewer.degree_program_preference)]
        for idx, row in filter_by_desired_course.iterrows():
            print(row['Name'])
            # Create interviewee instance if not existing
            interviewee_instance = Interviewee(name=row['Name'] , capes_id=['CAPES ID'], email="") if row['Name'] not in [interviewee.name for interviewee in booked_interviewees] else get_booked_interviewee(row['Name'], booked_interviewees)
            
            # get scores of interviewee/response
            scores_series = row[filter_by_desired_course.columns[filter_by_desired_course.columns.str.contains(f'{type} Slot {day}')]]
            melted_df = scores_series.to_frame().T.melt(
                ignore_index=False,
                var_name='Time Slot',
                value_name='Ranking'
            ).reset_index(drop=True).sort_values(by='Ranking', ascending=True)

            # clean scores to map to Timeslot enum values
            melted_df['Time Slot'] = melted_df['Time Slot'].str.replace(f'{type} Slot {day} - ', '', regex=False)
            melted_df_filtered = melted_df[melted_df['Ranking'] != 12].copy()
            melted_df_filtered['Time Slot Clean'] = melted_df_filtered['Time Slot'].str.strip('[]')
            # Create the reverse mapping dictionary: Time String -> Enum Member
            enum_map = {member.value: member for member in Times}
            melted_df_filtered['Time Enum Member'] = melted_df_filtered['Time Slot Clean'].map(enum_map)
            # map to interviewer with available timeslot
            for timeslot in melted_df_filtered['Time Enum Member']: 
                if(interviewer.add_to_interviewee_list(interviewee_instance, timeslot) == 1 and interviewee_instance.check_coinciding_times(timeslot, type, day) == 1):
                    interviewee_instance.add_time(timeslot, type=type, day=day)
                    filter_by_desired_course.drop(idx)
                    booked_interviewees.append(interviewee_instance) if interviewee_instance not in booked_interviewees else None
                    break
        filtered_df = filtered_df[~filtered_df['Name'].isin(booked_interviewees)]
    return filtered_df, booked_interviewees

def log_scheduler(day : str, type : str):
    for interviewer in interviewer_availability_dict[day][type]:
        print(f"----Interview Schedule for {interviewer.name}----")

        for timeslot in interviewer.time_slots:
            print(f"Timeslot: {timeslot}")
            try: 
                for interviewee in interviewer.interviewee_list[timeslot]:
                    print(interviewee)
            except KeyError:
                print("No interviewees for the timeslot.")
        print("---------------------")

### November 13 RC

In [48]:
carryover_interviewees, booked_interviewees = scheduling_algorithm(day='November 13', type='RC', filtered_df=filter_first_day_rc)

Cyress Lein Carpio
Marianne Veronica Punzalan
{<Times.Time1045_1115: '10:45-11:15 AM'>: []}
Clyde Lawrence Borrega
{<Times.Time1045_1115: '10:45-11:15 AM'>: [<__main__.Interviewee object at 0x7ca77adafb10>], <Times.Time1015_1045: '10:15-10:45 AM'>: []}
Xander Aia Danina Jaudalso
{<Times.Time1330_1400: '1:30-2:00 PM'>: []}
Leevan Hernandez 
{<Times.Time1330_1400: '1:30-2:00 PM'>: [<__main__.Interviewee object at 0x7ca77abd4c30>], <Times.Time1630_1700: '4:30-5:00 PM'>: []}
John Erickson Lao
{<Times.Time1330_1400: '1:30-2:00 PM'>: [<__main__.Interviewee object at 0x7ca77abd4c30>], <Times.Time1630_1700: '4:30-5:00 PM'>: [<__main__.Interviewee object at 0x7ca77abd48a0>], <Times.Time1045_1115: '10:45-11:15 AM'>: []}
John Rustly Mark Ambrocio


In [49]:
log_scheduler('November 13', 'RC')

----Interview Schedule for Huawei Technologies Phils. Inc.----
Timeslot: Times.Time0900_0930
No interviewees for the timeslot.
Timeslot: Times.Time0930_1000
No interviewees for the timeslot.
Timeslot: Times.Time1015_1045
Interviewee: Clyde Lawrence Borrega
Timeslot: Times.Time1045_1115
Interviewee: Marianne Veronica Punzalan
Timeslot: Times.Time1130_1200
No interviewees for the timeslot.
---------------------
----Interview Schedule for GHD----
Timeslot: Times.Time0900_0930
No interviewees for the timeslot.
Timeslot: Times.Time0930_1000
No interviewees for the timeslot.
Timeslot: Times.Time1015_1045
No interviewees for the timeslot.
Timeslot: Times.Time1045_1115
Interviewee: John Erickson Lao
Timeslot: Times.Time1130_1200
No interviewees for the timeslot.
Timeslot: Times.Time1330_1400
Interviewee: Xander Aia Danina Jaudalso
Timeslot: Times.Time1400_1430
No interviewees for the timeslot.
Timeslot: Times.Time1445_1515
No interviewees for the timeslot.
Timeslot: Times.Time1515_1545
No inte

## November 13 IS


In [50]:
filter_first_day_is = interviewee_df.sort_values(by='What is your most preferred date?  [Interview Simulation (IS)]', key= lambda col : col == 'November 13')

carryover_interviewees, booked_interviewees = scheduling_algorithm('November 13', 'IS', filter_first_day_is, booked_interviewees)

log_scheduler('November 13', 'IS')

Cyress Lein Carpio
Marianne Veronica Punzalan
{<Times.Time1515_1600: '3:15-4:00 PM'>: []}
Clyde Lawrence Borrega
{<Times.Time1515_1600: '3:15-4:00 PM'>: [<__main__.Interviewee object at 0x7ca77adafb10>], <Times.Time1000_1045: '10:00-10:45 AM'>: []}
Leevan Hernandez 
{<Times.Time1515_1600: '3:15-4:00 PM'>: [<__main__.Interviewee object at 0x7ca77adafb10>], <Times.Time1000_1045: '10:00-10:45 AM'>: [<__main__.Interviewee object at 0x7ca77a170cd0>], <Times.Time1415_1500: '2:15-3:00 PM'>: []}
John Erickson Lao
{<Times.Time1515_1600: '3:15-4:00 PM'>: [<__main__.Interviewee object at 0x7ca77adafb10>], <Times.Time1000_1045: '10:00-10:45 AM'>: [<__main__.Interviewee object at 0x7ca77a170cd0>], <Times.Time1415_1500: '2:15-3:00 PM'>: [<__main__.Interviewee object at 0x7ca77abd48a0>], <Times.Time1100_1145: '11:00-11:45 AM'>: []}
John Rustly Mark Ambrocio
Cyress Lein Carpio
Marianne Veronica Punzalan
Clyde Lawrence Borrega
{<Times.Time1000_1045: '10:00-10:45 AM'>: []}
----Interview Schedule for GHD

### Create Nov 13 RC Dataframe

In [51]:
fin_dict = [
]
for interviewer in interviewer_availability_dict['November 13']['RC']:
    for timeslot in interviewer.time_slots:
        try: 
            for idx, interviewee in enumerate(interviewer.interviewee_list[timeslot]):
                fin_dict.append([interviewer.name, timeslot.value, interviewee.name])
        except KeyError:
            fin_dict.append([interviewer.name, timeslot.value, 'None'])
# rc_nov_13_df = pd.DataFrame(
#     data = fin_dict
# )
final_df_nov_13_rc = pd.DataFrame(
    columns=['Company', 'Time Slot', 'Interviewee'],
    data=fin_dict
).set_index('Company')

final_df_nov_13_rc

,Time Slot,Interviewee
Company,,
Huawei Technologies Phils. Inc.,9:00-9:30 AM,None
Huawei Technologies Phils. Inc.,9:30-10:00 AM,None
Huawei Technologies Phils. Inc.,10:15-10:45 AM,Clyde Lawrence Borrega
Huawei Technologies Phils. Inc.,10:45-11:15 AM,Marianne Veronica Punzalan
Huawei Technologies Phils. Inc.,11:30 AM-12:00 PM,None
GHD,9:00-9:30 AM,None
GHD,9:30-10:00 AM,None
GHD,10:15-10:45 AM,None
GHD,10:45-11:15 AM,John Erickson Lao
